In [3]:
# Cellule 1 : imports, graine, chemins
import os, glob, json, random
import numpy as np, pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device :", dev)

hits = glob.glob("/kaggle/input/**/train.tsv", recursive=True)
assert hits, "train.tsv introuvable : ajoute ton dataset avec Add Input"
P = os.path.dirname(hits[0]) + "/"

def has_jpg(d):
    return os.path.isdir(d) and any(f.endswith(".jpg") for f in os.listdir(d)[:20])

IMG = next((d for d in (P + "images/images/", P + "images/") if has_jpg(d)), None)
assert IMG, "dossier d'images introuvable dans " + P

OUT = "/kaggle/working/"
os.makedirs(OUT + "emb", exist_ok=True)
os.makedirs(OUT + "results", exist_ok=True)
print("P   =", P)
print("IMG =", IMG)

device : cuda
P   = /kaggle/input/datasets/sahnounsahnoun/data-fakenewes/
IMG = /kaggle/input/datasets/sahnounsahnoun/data-fakenewes/images/images/


In [4]:
# Cellule 2 : chargement des splits et controles d'integrite
S = ["train", "val", "test"]
df = {s: pd.read_csv(P + s + ".tsv", sep="\t") for s in S}

have = {f[:-4] for f in os.listdir(IMG)}
for s in S:
    miss = int((~df[s]["id"].astype(str).isin(have)).sum())
    print(s, len(df[s]), "exemples | images manquantes :", miss)
    assert miss == 0, "images manquantes : verifie la version du dataset"

y = {s: df[s]["2_way_label"].values for s in S}
y6 = {s: df[s]["6_way_label"].values for s in S}

# les probabilites d'E1 doivent correspondre ligne par ligne aux splits
e1 = {s: np.load(P + "e1_" + s + "_proba.npy") for s in ["val", "test"]}
for s in ["val", "test"]:
    assert len(e1[s]) == len(df[s]), (
        "e1_%s_proba.npy a %d lignes, %s.tsv en a %d : mets a jour le dataset "
        "avec les .npy regeneres apres le nettoyage" % (s, len(e1[s]), s, len(df[s])))
print("alignement E1 / splits : OK")

train 7432 exemples | images manquantes : 0
val 1403 exemples | images manquantes : 0
test 1396 exemples | images manquantes : 0
alignement E1 / splits : OK


In [5]:
# Cellule 3 : embeddings image, ResNet-18 gele (512 dimensions)
tf = transforms.Compose([transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

class Imgs(Dataset):
    def __init__(self, ids):
        self.ids = list(ids)
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        return tf(Image.open(IMG + self.ids[i] + ".jpg").convert("RGB"))

resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet.fc = torch.nn.Identity()
resnet = resnet.to(dev).eval()

def img_emb(ids):
    out = []
    for b in DataLoader(Imgs(ids), batch_size=64, shuffle=False, num_workers=2):
        with torch.no_grad():
            out.append(resnet(b.to(dev)).cpu().numpy())
    return np.concatenate(out).astype("float32")

Xi = {}
for s in S:
    Xi[s] = img_emb(df[s]["id"].astype(str))
    np.save(OUT + "emb/img_" + s + ".npy", Xi[s])
    print("image", s, Xi[s].shape)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 190MB/s] 


image train (7432, 512)
image val (1403, 512)
image test (1396, 512)


In [6]:
# Cellule 4 : embeddings texte, DistilBERT gele (768 dimensions, moyenne des tokens)
from transformers import AutoTokenizer, AutoModel

tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
bert = AutoModel.from_pretrained("distilbert-base-uncased").to(dev).eval()

def txt_emb(texts):
    texts = list(texts)
    out = []
    for i in range(0, len(texts), 64):
        t = tok(texts[i:i + 64], padding=True, truncation=True, max_length=64, return_tensors="pt").to(dev)
        with torch.no_grad():
            h = bert(**t).last_hidden_state
        m = t["attention_mask"].unsqueeze(-1)
        out.append(((h * m).sum(1) / m.sum(1)).cpu().numpy())
    return np.concatenate(out).astype("float32")

Xt = {}
for s in S:
    Xt[s] = txt_emb(df[s]["clean_title"].astype(str))
    np.save(OUT + "emb/txt_" + s + ".npy", Xt[s])
    print("texte", s, Xt[s].shape)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


texte train (7432, 768)
texte val (1403, 768)
texte test (1396, 768)


In [7]:
# Cellule 5 : fonctions d'evaluation et d'entrainement (regression logistique sur embeddings)
GRID = [0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1]

def metrics(y_true, p):
    pred = (p > 0.5).astype(int)
    return dict(acc=accuracy_score(y_true, pred),
                prec=precision_score(y_true, pred, average="macro", zero_division=0),
                rec=recall_score(y_true, pred, average="macro", zero_division=0),
                macro_f1=f1_score(y_true, pred, average="macro", zero_division=0),
                auc=roc_auc_score(y_true, p))

def run_lr(name, X):
    mk = lambda C: make_pipeline(StandardScaler(), LogisticRegression(
        C=C, max_iter=3000, class_weight="balanced", random_state=SEED))
    # choix de C sur la validation uniquement
    val = {C: f1_score(y["val"], mk(C).fit(X["train"], y["train"]).predict(X["val"]), average="macro")
           for C in GRID}
    best = max(val, key=val.get)
    print(name, "Macro F1 validation par C :", {c: round(v, 4) for c, v in val.items()})
    if best in (GRID[0], GRID[-1]):
        print("attention : le meilleur C est en bord de grille")
    clf = mk(best).fit(X["train"], y["train"])
    pv = clf.predict_proba(X["val"])[:, 1]
    pt = clf.predict_proba(X["test"])[:, 1]
    m = metrics(y["test"], pt)
    print(name, "meilleur C =", best, "| val F1 =", round(val[best], 4),
          "| TEST", {k: round(float(v), 4) for k, v in m.items()})
    np.save(OUT + "results/" + name.lower() + "_val_proba.npy", pv)
    np.save(OUT + "results/" + name.lower() + "_test_proba.npy", pt)
    return dict(best_C=best, val_macro_f1=val[best], test=m), pt

In [8]:
# Cellule 6 : E3 (image seule) et E2 (texte seul, DistilBERT gele)
res, proba = {}, {}
res["E3"], proba["E3"] = run_lr("E3", Xi)
res["E2"], proba["E2"] = run_lr("E2", Xt)
proba["E1"] = e1["test"]   # probabilites d'E1 (TF-IDF), deja calculees sur le PC

E3 Macro F1 validation par C : {0.0003: 0.6711, 0.001: 0.6804, 0.003: 0.682, 0.01: 0.6799, 0.03: 0.6743, 0.1: 0.6686, 0.3: 0.6657, 1: 0.665}
E3 meilleur C = 0.003 | val F1 = 0.682 | TEST {'acc': 0.6991, 'prec': 0.7001, 'rec': 0.6989, 'macro_f1': 0.6986, 'auc': 0.7547}
E2 Macro F1 validation par C : {0.0003: 0.796, 0.001: 0.8018, 0.003: 0.8068, 0.01: 0.8068, 0.03: 0.8082, 0.1: 0.8104, 0.3: 0.8075, 1: 0.8096}
E2 meilleur C = 0.1 | val F1 = 0.8104 | TEST {'acc': 0.8009, 'prec': 0.8009, 'rec': 0.8008, 'macro_f1': 0.8008, 'auc': 0.8615}


In [9]:
# Cellule 7 : tableau comparatif sur le test, et precision par type de faux
tab = pd.DataFrame({
    "Majoritaire (tout vrai)": metrics(y["test"], np.ones(len(y["test"]))),
    "E1 texte TF-IDF": metrics(y["test"], proba["E1"]),
    "E2 texte DistilBERT": res["E2"]["test"],
    "E3 image ResNet-18": res["E3"]["test"],
}).T[["acc", "macro_f1", "auc"]].astype(float).round(4)
print(tab, "\n")

names = {0: "vrai", 1: "satire", 2: "trompeur", 3: "bots", 4: "image manipulee", 5: "fausse connexion"}
per_type = pd.DataFrame({"n": pd.Series(y6["test"]).value_counts().sort_index()})
for k in ["E1", "E2", "E3"]:
    ok = (proba[k] > 0.5).astype(int) == y["test"]
    per_type[k] = pd.Series(ok).groupby(y6["test"]).mean().round(3)
per_type.index = [str(i) + " " + names.get(i, "?") for i in per_type.index]
print("Precision par type (test) :")
print(per_type)

# diagnostic du raccourci : predire avec le seul subreddit (ce n'est PAS un modele a comparer)
maj = df["train"].groupby("subreddit")["2_way_label"].agg(lambda s: s.mode()[0])
sp = df["test"]["subreddit"].map(maj)
print("\nlignes de test dont le subreddit est absent du train :", int(sp.isna().sum()))
sp = sp.fillna(1).astype(int)
sub_diag = dict(acc=accuracy_score(y["test"], sp), macro_f1=f1_score(y["test"], sp, average="macro"))
print("Diagnostic : le subreddit seul donne", {k: round(v, 4) for k, v in sub_diag.items()})

                            acc  macro_f1     auc
Majoritaire (tout vrai)  0.5029    0.3346  0.5000
E1 texte TF-IDF          0.7507    0.7502  0.8218
E2 texte DistilBERT      0.8009    0.8008  0.8615
E3 image ResNet-18       0.6991    0.6986  0.7547 

Precision par type (test) :
                      n     E1     E2     E3
0 vrai              702  0.791  0.803  0.736
1 satire             63  0.540  0.603  0.587
2 trompeur          212  0.618  0.675  0.486
3 bots               21  0.238  0.476  0.333
4 image manipulee   347  0.833  0.942  0.778
5 fausse connexion   51  0.667  0.706  0.824

lignes de test dont le subreddit est absent du train : 0
Diagnostic : le subreddit seul donne {'acc': 1.0, 'macro_f1': 1.0}


In [10]:
# Cellule 8 : sauvegarde des metriques et liste des fichiers produits
log = dict(seed=SEED, device=dev,
           n={s: len(df[s]) for s in S},
           E1=metrics(y["test"], proba["E1"]), E2=res["E2"], E3=res["E3"],
           table=tab.to_dict(), per_type=per_type.to_dict(), subreddit_diagnostic=sub_diag)
with open(OUT + "results/e2_e3_metrics.json", "w", encoding="utf-8") as f:
    json.dump(log, f, indent=2, ensure_ascii=False,
              default=lambda o: o.item() if hasattr(o, "item") else str(o))

for root, _, files in os.walk(OUT):
    for fn in sorted(files):
        p = os.path.join(root, fn)
        print(p, round(os.path.getsize(p) / 1e6, 2), "Mo")

/kaggle/working/emb/img_test.npy 2.86 Mo
/kaggle/working/emb/img_train.npy 15.22 Mo
/kaggle/working/emb/img_val.npy 2.87 Mo
/kaggle/working/emb/txt_test.npy 4.29 Mo
/kaggle/working/emb/txt_train.npy 22.83 Mo
/kaggle/working/emb/txt_val.npy 4.31 Mo
/kaggle/working/.virtual_documents/__notebook_source__.ipynb 0.01 Mo
/kaggle/working/results/e2_e3_metrics.json 0.0 Mo
/kaggle/working/results/e2_test_proba.npy 0.01 Mo
/kaggle/working/results/e2_val_proba.npy 0.01 Mo
/kaggle/working/results/e3_test_proba.npy 0.01 Mo
/kaggle/working/results/e3_val_proba.npy 0.01 Mo
